In [9]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "etf_adjusted_close.csv"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

prices = pd.read_csv(
    DATA_PATH,
    index_col="Date",
    parse_dates=True,
)

returns = prices.pct_change(fill_method=None).dropna()

#returns.head()

,SPY,QQQ,IWM,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ
Date,,,,,,,,,,,,
2007-04-12,0.004444,0.007917,0.006725,0.006677,0.016719,0.000970,0.000228,0.001505,0.000671,-0.001342,0.005903,-0.006805
2007-04-13,0.004562,0.002020,0.006432,0.003316,0.004909,-0.001697,-0.003089,-0.001690,-0.001820,0.012688,0.005477,0.011292
2007-04-16,0.009496,0.009182,0.013767,0.010552,0.011886,0.001093,0.005510,0.000847,-0.000384,0.008255,-0.007782,0.001255
2007-04-17,0.002658,0.002219,-0.003031,0.000126,-0.005391,0.004246,0.005594,0.005452,-0.000480,-0.005848,-0.010196,0.013033
2007-04-18,0.001224,-0.003322,-0.005837,-0.000377,-0.007442,0.002295,0.004995,0.000280,0.000288,0.005588,0.000396,-0.006185


In [3]:
benchmark_weights = pd.DataFrame(
    0.0,
    index=["SPY", "60SPY_40IEF", "Equal_Weight"],
    columns=returns.columns,
)

benchmark_weights.loc['SPY','SPY'] = 1
benchmark_weights.loc['60SPY_40IEF', ['SPY', 'IEF']] = [0.6, 0.4]
benchmark_weights.loc['Equal_Weight'] = 1/len(returns.columns)

#benchmark_weights.sum(axis=1)
benchmark_weights.head()

,SPY,QQQ,IWM,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ
SPY,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60SPY_40IEF,0.600000,0.000000,0.000000,0.000000,0.000000,0.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Equal_Weight,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333


In [17]:
# Select the last available trading day of each month as a portfolio formation date.

estimation_start = returns.index.min()
target_start = estimation_start + pd.DateOffset(years=3)

backtest_start = returns.index[
    returns.index >= target_start
][0]

backtest_asset_returns = returns.loc[backtest_start:]
backtest_asset_prices = prices.loc[backtest_start:]

#print(backtest_asset_prices.head())

monthly_formation_dates = (
    backtest_asset_returns
    .loc[:"2020-03-31"]
    .groupby(
        backtest_asset_returns
        .loc[:"2020-03-31"]
        .index
        .to_period("M")
    )
    .tail(1)
    .index
)

# Treat backtest_start as the initial formation date.
formation_dates = monthly_formation_dates.insert(
    0,
    backtest_start,
).drop_duplicates()

benchmark_wealth = {}

for benchmark_name in benchmark_weights.index:

    target_weights = benchmark_weights.loc[
        benchmark_name
    ].copy()

    current_wealth = 1.0

    wealth_segments = [
        pd.Series(
            [current_wealth],
            index=[formation_dates[0]],
            name=benchmark_name,
        )
    ]

    for formation_date, next_formation_date in zip(
        formation_dates[:-1],
        formation_dates[1:],
    ):

        buy_in_prices = backtest_asset_prices.loc[
            formation_date
        ]

        holding_period_prices = backtest_asset_prices.loc[
            (backtest_asset_prices.index > formation_date)
            & (
                backtest_asset_prices.index
                <= next_formation_date
            )
        ]

        relative_prices = holding_period_prices.div(
            buy_in_prices,
            axis="columns",
        )

        relative_portfolio_value = relative_prices.dot(
            target_weights
        )

        segment_wealth = (
            current_wealth
            * relative_portfolio_value
        )

        segment_wealth.name = benchmark_name
        wealth_segments.append(segment_wealth)

        current_wealth = segment_wealth.iloc[-1]

    benchmark_wealth[benchmark_name] = pd.concat(
        wealth_segments
    )

benchmark_wealth = pd.DataFrame(benchmark_wealth)

benchmark_wealth.head(5)

,SPY,60SPY_40IEF,Equal_Weight
2010-04-12,1.000000,1.000000,1.000000
2010-04-13,1.000752,1.001212,1.003501
2010-04-14,1.012109,1.006952,1.010255
2010-04-15,1.012945,1.008125,1.009470
2010-04-16,0.996826,1.000873,0.997039


In [29]:
#calculating statistics for 2010-2020 period. 
import math
number_of_days = benchmark_wealth.shape[0]
benchmark_returns = benchmark_wealth.pct_change(fill_method=None).dropna()

annualized_returns = {}
annualized_volatilities = {}
sharpe_ratios = {}
var_95 = {}
cvar_95 = {}
max_drawdown = {}
final_wealths = {}

for benchmark_name in benchmark_returns.columns:
    portfolio_returns = benchmark_returns[benchmark_name]

    initial_wealth = benchmark_wealth[benchmark_name].iloc[0]
    ending_wealth = benchmark_wealth[benchmark_name].iloc[-1]

    annualized_returns[benchmark_name] = (
        (ending_wealth / initial_wealth)
        ** (252 / number_of_days)
        - 1
    )

    annualized_volatilities[benchmark_name] = (
        portfolio_returns.std() * np.sqrt(252)
    )

    sharpe_ratios[benchmark_name] = (
        portfolio_returns.mean()
        / portfolio_returns.std()
        * np.sqrt(252)
    )

    var_95[benchmark_name] = portfolio_returns.quantile(0.05)

    cvar_95[benchmark_name] = portfolio_returns[
        portfolio_returns <= var_95[benchmark_name]
    ].mean()

    max_drawdown[benchmark_name] = (benchmark_wealth[benchmark_name] / benchmark_wealth[benchmark_name].cummax()).min() - 1

    final_wealths[benchmark_name] = ending_wealth

benchmark_summary = pd.DataFrame({
    "annualized_return": annualized_returns,
    "annualized_volatility": annualized_volatilities,
    "sharpe_ratio": sharpe_ratios,
    "VaR_95": var_95,
    "CVaR_95": cvar_95,
    "Max_drawdown": max_drawdown,
    "final_wealth": final_wealths,
})

benchmark_summary.round(4)



,annualized_return,annualized_volatility,sharpe_ratio,VaR_95,CVaR_95,Max_drawdown,final_wealth
SPY,0.1021,0.1689,0.6606,-0.0161,-0.0264,-0.3372,2.6337
60SPY_40IEF,0.0859,0.0900,0.9611,-0.0081,-0.0138,-0.1889,2.2733
Equal_Weight,0.0573,0.0993,0.6118,-0.0088,-0.0148,-0.2218,1.7431


### Benchmark Portfolio Comparison

Relative to the 100% SPY benchmark, the 60/40 SPY–IEF portfolio sacrifices annualized return and final wealth in exchange for lower overall and downside risk.

Its lower annualized volatility indicates less variability in daily returns. Its smaller 95% VaR and CVaR imply lower losses during adverse market conditions, with CVaR specifically measuring the average loss among the worst 5% of trading days. The 60/40 portfolio also has a smaller maximum drawdown, meaning that its largest peak-to-trough wealth decline is less severe.

Despite its lower absolute return, the 60/40 portfolio achieves a higher Sharpe ratio. Therefore, it produces more return per unit of volatility and offers better risk-adjusted performance under the assumption of a zero risk-free rate.

Overall, adding intermediate-term Treasury exposure through IEF reduces both ordinary return fluctuations and severe downside risk. The cost of this protection is lower long-term wealth accumulation compared with holding 100% SPY.